# 10 prepare infercnvpy inputs

**Purpose:** Prepare ordered-gene inputs for infercnvpy strategies.

**Original analysis notebook:** `10_prepare_CNV_inputs_HVG8000_r0_6_fullgene.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 26 — Prepare CNV inference inputs — HVG8000 r0.6 full-gene pipeline

This notebook prepares **principled CNV inputs** from the marker-based annotated full-gene object.

Main goals:

1. Use the **full-gene object** with 15,176 genes, not the HVG object.
2. Preserve raw counts and place them in `adata.X` for CNV tools.
3. Add gene genomic positions from GENCODE v49.
4. Define CNV query/reference groups based on annotation + bias QC.
5. Create multiple reference strategies for robustness:
   - `core_conservative`
   - `core_plus_lymphoid`
   - `extended_myeloid`
   - `review_cycling`
   - `review_ambiguous`
6. Export inferCNV/infercnvpy-ready files and sampled h5ad subsets.

Important: tumor-like labels are still **putative**. Malignancy should be confirmed by CNV, not assumed from markers alone.


In [ ]:
# =========================
# Imports and global settings
# =========================

from pathlib import Path
import os
import re
import gzip
import warnings

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import anndata as ad

warnings.filterwarnings("ignore")

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RAW_DIR = PROJECT_DIR / "data" / "interim"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures"
REFERENCE_DIR = PROJECT_DIR / "data" / "reference"
CNV_DIR = RAW_DIR / "integrated" / "CNV_inputs"

for d in [PROCESSED_DIR, RAW_DIR, METADATA_DIR, FIGURES_DIR, REFERENCE_DIR, CNV_DIR]:
    d.mkdir(parents=True, exist_ok=True)

INPUT_H5AD = PROCESSED_DIR / "integrated" / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status.h5ad"
FALLBACK_H5AD = PROCESSED_DIR / "integrated" / "GBM_core_4datasets_fullgene_marker_based_annotation.h5ad"
GTF_GZ = REFERENCE_DIR / "gencode.v49.primary_assembly.annotation.gtf.gz"

CLUSTER_KEY = "leiden_scvi_main"
ANNOTATION_COL = "annotation_marker_based_HVG8000_r0_6"
LINEAGE_COL = "lineage_marker_based_HVG8000_r0_6"
CONFIDENCE_COL = "annotation_confidence_HVG8000_r0_6"
CNV_REQUIRED_COL = "requires_CNV_for_malignancy_call_HVG8000_r0_6"
BIAS_STATUS_COL = "cluster_bias_status_HVG8000_r0_6"
BIAS_REASON_COL = "cluster_bias_reason_HVG8000_r0_6"
CNV_ROLE_COL = "cnv_role_recommendation_HVG8000_r0_6"

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

MAX_REF_PER_STRATEGY = 20000
MAX_QUERY_PER_STRATEGY = 25000
MAX_PER_CLUSTER_REF = 5000
MAX_PER_CLUSTER_QUERY = 7000

print("Project:", PROJECT_DIR)
print("Input h5ad:", INPUT_H5AD)
print("GTF:", GTF_GZ)
print("CNV output dir:", CNV_DIR)


In [ ]:
# =========================
# Helper functions
# =========================

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")


def write_text_replace(lines, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    with open(path, "w") as f:
        for line in lines:
            f.write(str(line).rstrip("\n") + "\n")
    print(f"[SAVED] {path} n_lines={len(lines)}")


def write_h5ad_replace(adata_obj, path, compression="gzip"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    adata_obj.write_h5ad(path, compression=compression)
    print(f"[SAVED] {path} shape={adata_obj.shape}")


def ensure_string_obs(adata_obj, cols):
    for col in cols:
        if col in adata_obj.obs.columns:
            adata_obj.obs[col] = adata_obj.obs[col].astype(str).fillna("unknown")


def sparse_integer_like(X, n_rows=2000, n_cols=2000):
    sub = X[:min(X.shape[0], n_rows), :min(X.shape[1], n_cols)]
    if sp.issparse(sub):
        vals = sub.data
    else:
        vals = np.asarray(sub).ravel()
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return True, None, None, False
    return bool(np.allclose(vals, np.round(vals))), float(vals.min()), float(vals.max()), bool(np.any(vals < 0))


def sample_cells_by_cluster(obs, clusters, cluster_col, max_total, max_per_cluster, seed=42):
    rng = np.random.default_rng(seed)
    chosen = []
    obs2 = obs[obs[cluster_col].astype(str).isin([str(c) for c in clusters])]
    for cl in sorted(obs2[cluster_col].astype(str).unique(), key=lambda x: int(x) if str(x).isdigit() else str(x)):
        idx = obs2.index[obs2[cluster_col].astype(str) == str(cl)].to_numpy()
        n = min(len(idx), max_per_cluster)
        if n > 0:
            chosen.extend(rng.choice(idx, size=n, replace=False).tolist())
    if len(chosen) > max_total:
        chosen = rng.choice(np.array(chosen), size=max_total, replace=False).tolist()
    return list(chosen)


def parse_gtf_gene_positions(gtf_gz_path):
    rows = []
    if not Path(gtf_gz_path).exists():
        raise FileNotFoundError(f"GTF not found: {gtf_gz_path}")
    opener = gzip.open if str(gtf_gz_path).endswith(".gz") else open
    with opener(gtf_gz_path, "rt") as f:
        for line in f:
            if not line or line.startswith("#"):
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 9:
                continue
            chrom, source, feature, start, end, score, strand, frame, attrs = parts
            if feature != "gene":
                continue
            gene_id = None
            gene_name = None
            gene_type = None
            for m in re.finditer(r'(gene_id|gene_name|gene_type|gene_biotype) "([^"]+)"', attrs):
                key, val = m.group(1), m.group(2)
                if key == "gene_id":
                    gene_id = val.split(".")[0]
                elif key == "gene_name":
                    gene_name = val
                elif key in ["gene_type", "gene_biotype"]:
                    gene_type = val
            if gene_name is None:
                continue
            rows.append({
                "gene_symbol": gene_name,
                "gene_id": gene_id,
                "chromosome": chrom.replace("chr", ""),
                "start": int(start),
                "end": int(end),
                "strand": strand,
                "gene_type": gene_type,
            })
    gtf = pd.DataFrame(rows)
    chrom_order = {str(i): i for i in range(1, 23)}
    chrom_order.update({"X": 23, "Y": 24, "M": 25, "MT": 25})
    gtf["chrom_order"] = gtf["chromosome"].map(chrom_order).fillna(999).astype(int)
    gtf["protein_coding_rank"] = (gtf["gene_type"] == "protein_coding").astype(int)
    gtf = gtf.sort_values(["gene_symbol", "protein_coding_rank", "chrom_order", "start"], ascending=[True, False, True, True])
    gtf = gtf.drop_duplicates("gene_symbol", keep="first").copy()
    return gtf

print("Helpers ready")


## 1. Read annotated+bias full-gene object

This object must be the **full-gene** annotated object with raw counts preserved. It should have 351,427 cells and about 15,176 genes.


In [ ]:
# =========================
# Read full-gene annotated object
# =========================

if INPUT_H5AD.exists():
    input_path = INPUT_H5AD
elif FALLBACK_H5AD.exists():
    input_path = FALLBACK_H5AD
    print(f"[WARNING] Bias-status object not found. Using fallback annotated object: {input_path}")
else:
    raise FileNotFoundError(f"Neither input exists:\n{INPUT_H5AD}\n{FALLBACK_H5AD}")

adata = sc.read_h5ad(input_path)
print(adata)
print("layers:", list(adata.layers.keys()))
print("obsm:", list(adata.obsm.keys()))
print("obs columns:", list(adata.obs.columns)[:80])

required_obs = [CLUSTER_KEY, ANNOTATION_COL, LINEAGE_COL, CONFIDENCE_COL]
missing_required = [c for c in required_obs if c not in adata.obs.columns]
if missing_required:
    raise KeyError(f"Missing required obs columns: {missing_required}")

ensure_string_obs(adata, [
    CLUSTER_KEY, ANNOTATION_COL, LINEAGE_COL, CONFIDENCE_COL,
    CNV_REQUIRED_COL, BIAS_STATUS_COL, BIAS_REASON_COL, CNV_ROLE_COL,
    "core_dataset", "dataset_id", "condition", "patient_id", "sample_id", "scvi_batch"
])

counts_source = None
if "counts" in adata.layers:
    counts_source = "adata.layers['counts']"
    adata.X = adata.layers["counts"].copy()
else:
    counts_source = "adata.X"
    print("[WARNING] No counts layer found. Assuming adata.X contains raw counts.")

integer_like, x_min, x_max, any_negative = sparse_integer_like(adata.X)
print("counts_source:", counts_source)
print("X integer-like:", integer_like)
print("X min/max:", x_min, x_max)
print("any negative:", any_negative)

if not integer_like or any_negative:
    raise ValueError("CNV input requires raw non-negative count-like expression in adata.X.")

readiness = pd.DataFrame([{
    "input_h5ad": str(input_path),
    "n_cells": adata.n_obs,
    "n_genes": adata.n_vars,
    "counts_source_for_cnv": counts_source,
    "has_counts_layer": "counts" in adata.layers,
    "x_integer_like": integer_like,
    "x_min_sample": x_min,
    "x_max_sample": x_max,
    "any_negative_sample": any_negative,
    "cluster_key": CLUSTER_KEY,
    "annotation_col": ANNOTATION_COL,
    "lineage_col": LINEAGE_COL,
    "confidence_col": CONFIDENCE_COL,
    "bias_status_col_present": BIAS_STATUS_COL in adata.obs.columns,
    "cnv_role_col_present": CNV_ROLE_COL in adata.obs.columns,
}])

READINESS_TSV = METADATA_DIR / "CNV_input_readiness_report_HVG8000_r0_6.tsv"
write_tsv_replace(readiness, READINESS_TSV)
display(readiness)


## 2. Define CNV categories and reference strategies

The definitions below follow the current annotation and bias/QC results. The main tumor-like calls are still **putative** and must be validated by CNV.


In [ ]:
# =========================
# CNV cluster roles and strategies
# =========================

query_tumor_like = ["0", "4", "7", "11"]
review_cycling = ["2"]
review_ambiguous = ["9", "12"]
exclude_or_low_conf = ["16", "17"]

core_reference = ["8", "14"]
lymphoid_reference = ["5", "15"]
myeloid_reference = ["1", "3", "6"]
myeloid_review = ["10"]
stromal_review = ["13"]

role_map = {}
for c in query_tumor_like:
    role_map[c] = "query_putative_tumor_like"
for c in review_cycling:
    role_map[c] = "query_review_cycling"
for c in review_ambiguous:
    role_map[c] = "query_review_ambiguous"
for c in exclude_or_low_conf:
    role_map[c] = "exclude_or_low_confidence"
for c in core_reference:
    role_map[c] = "reference_core_conservative"
for c in lymphoid_reference:
    role_map[c] = "reference_lymphoid_candidate"
for c in myeloid_reference:
    role_map[c] = "reference_myeloid_extended_candidate"
for c in myeloid_review:
    role_map[c] = "reference_myeloid_review_not_core"
for c in stromal_review:
    role_map[c] = "reference_stromal_review_not_core"

adata.obs["cnv_cluster_role_HVG8000_r0_6"] = adata.obs[CLUSTER_KEY].astype(str).map(role_map).fillna("unassigned_review")

strategies = {
    "core_conservative": {"reference_clusters": core_reference, "query_clusters": query_tumor_like, "description": "Most conservative CNV run: oligodendrocyte and endothelial references only; putative tumor-like query clusters only."},
    "core_plus_lymphoid": {"reference_clusters": core_reference + lymphoid_reference, "query_clusters": query_tumor_like, "description": "Conservative reference plus lymphoid clusters; useful robustness check, but lymphoid clusters are DS3-biased."},
    "extended_myeloid": {"reference_clusters": core_reference + myeloid_reference, "query_clusters": query_tumor_like, "description": "Extended reference including broadly represented myeloid/TAM clusters; excludes patient-dominant myeloid cluster 10."},
    "review_cycling": {"reference_clusters": core_reference, "query_clusters": review_cycling, "description": "Review run for cycling cluster to decide whether cycling cells are tumor-like/malignant or normal/immune cycling."},
    "review_ambiguous": {"reference_clusters": core_reference, "query_clusters": review_ambiguous, "description": "Review run for ambiguous mixed/MES-stromal-like clusters."},
}

strategy_rows = []
for name, info in strategies.items():
    strategy_rows.append({
        "strategy": name,
        "reference_clusters": ",".join(info["reference_clusters"]),
        "query_clusters": ",".join(info["query_clusters"]),
        "description": info["description"],
        "n_reference_clusters": len(info["reference_clusters"]),
        "n_query_clusters": len(info["query_clusters"]),
    })
strategy_df = pd.DataFrame(strategy_rows)
STRATEGY_TSV = METADATA_DIR / "CNV_reference_strategy_definitions_HVG8000_r0_6.tsv"
write_tsv_replace(strategy_df, STRATEGY_TSV)
display(strategy_df)


In [ ]:
# =========================
# Cluster and cell counts by CNV role
# =========================

cluster_role_counts = (
    adata.obs
    .groupby([CLUSTER_KEY, "cnv_cluster_role_HVG8000_r0_6", ANNOTATION_COL, LINEAGE_COL, CONFIDENCE_COL], observed=True)
    .size()
    .reset_index(name="n_cells")
    .sort_values(CLUSTER_KEY, key=lambda s: s.astype(str).map(lambda x: int(x) if x.isdigit() else x))
)
if BIAS_STATUS_COL in adata.obs.columns:
    bias_by_cluster = adata.obs[[CLUSTER_KEY, BIAS_STATUS_COL]].drop_duplicates().copy()
    cluster_role_counts = cluster_role_counts.merge(bias_by_cluster, on=CLUSTER_KEY, how="left")
role_cell_counts = adata.obs["cnv_cluster_role_HVG8000_r0_6"].value_counts().rename_axis("cnv_cluster_role").reset_index(name="n_cells")
role_cell_counts["fraction_cells"] = role_cell_counts["n_cells"] / adata.n_obs
CLUSTER_CATEGORY_TSV = METADATA_DIR / "CNV_input_cluster_category_counts_HVG8000_r0_6.tsv"
CELL_CATEGORY_TSV = METADATA_DIR / "CNV_input_cell_category_counts_HVG8000_r0_6.tsv"
write_tsv_replace(cluster_role_counts, CLUSTER_CATEGORY_TSV)
write_tsv_replace(role_cell_counts, CELL_CATEGORY_TSV)
display(cluster_role_counts)
display(role_cell_counts)


## 3. Parse gene positions from GENCODE and subset to gene-ordered genes

CNV tools need genes ordered by chromosome and genomic coordinate. This step matches the 15,176 full-gene symbols to GENCODE v49.


In [ ]:
# =========================
# Parse GTF and match gene positions
# =========================

gtf_pos = parse_gtf_gene_positions(GTF_GZ)
print("GTF gene symbols:", gtf_pos.shape)
var_df = pd.DataFrame({"gene_symbol": adata.var_names.astype(str)})
matched = var_df.merge(gtf_pos, on="gene_symbol", how="left")
matched["is_matched"] = matched["chromosome"].notna()
canonical_chroms = [str(i) for i in range(1, 23)] + ["X", "Y"]
matched["is_canonical"] = matched["chromosome"].astype(str).isin(canonical_chroms)
matched["usable_for_cnv_gene_order"] = matched["is_matched"] & matched["is_canonical"]
chrom_order = {str(i): i for i in range(1, 23)}
chrom_order.update({"X": 23, "Y": 24})
matched["chrom_order"] = matched["chromosome"].map(chrom_order).fillna(999).astype(int)
matched = matched.sort_values(["usable_for_cnv_gene_order", "chrom_order", "start", "gene_symbol"], ascending=[False, True, True, True]).copy()
usable = matched[matched["usable_for_cnv_gene_order"]].copy()
usable_genes = usable["gene_symbol"].tolist()
position_report = pd.DataFrame([{
    "n_genes_total": adata.n_vars,
    "n_genes_matched_to_gtf": int(matched["is_matched"].sum()),
    "n_genes_canonical_usable": int(matched["usable_for_cnv_gene_order"].sum()),
    "fraction_matched_to_gtf": float(matched["is_matched"].mean()),
    "fraction_canonical_usable": float(matched["usable_for_cnv_gene_order"].mean()),
    "gtf_file": str(GTF_GZ),
}])
GENE_POSITION_REPORT_TSV = METADATA_DIR / "CNV_gene_position_readiness_report_HVG8000_r0_6.tsv"
GENE_MATCH_TSV = METADATA_DIR / "CNV_gene_order_gencode_v49_matched_header_HVG8000_r0_6.tsv"
GENE_ORDER_NOHEADER_TSV = METADATA_DIR / "CNV_gene_order_gencode_v49_matched_noheader_HVG8000_r0_6.tsv"
GENE_UNMATCHED_TSV = METADATA_DIR / "CNV_gene_order_unmatched_genes_HVG8000_r0_6.tsv"
write_tsv_replace(position_report, GENE_POSITION_REPORT_TSV)
infercnv_gene_order = usable[["gene_symbol", "chromosome", "start", "end"]].copy()
infercnv_gene_order.columns = ["gene", "chr", "start", "end"]
write_tsv_replace(infercnv_gene_order, GENE_MATCH_TSV)
GENE_ORDER_NOHEADER_TSV.parent.mkdir(parents=True, exist_ok=True)
if GENE_ORDER_NOHEADER_TSV.exists():
    GENE_ORDER_NOHEADER_TSV.unlink()
infercnv_gene_order.to_csv(GENE_ORDER_NOHEADER_TSV, sep="\t", index=False, header=False)
print(f"[SAVED] {GENE_ORDER_NOHEADER_TSV} shape={infercnv_gene_order.shape}")
unmatched = matched[~matched["usable_for_cnv_gene_order"]].copy()
write_tsv_replace(unmatched[["gene_symbol", "is_matched", "chromosome", "start", "end", "gene_type"]], GENE_UNMATCHED_TSV)
display(position_report)
display(infercnv_gene_order.head())
display(unmatched.head())


## 4. Save all-cell CNV input objects

Two all-cell objects are saved:

1. `allcells_countsX`: all 15,176 genes, raw counts in `X`.
2. `gene_ordered_countsX`: only genes with canonical genomic coordinates, ordered by genomic position.

Both can be used for downstream CNV input preparation. The gene-ordered object is more suitable for infercnvpy-like workflows.


In [ ]:
# =========================
# Save all-cell CNV h5ad objects
# =========================

ALLCELLS_H5AD = CNV_DIR / "GBM_core_4datasets_CNV_input_HVG8000_r0_6_allcells_fullgene_countsX.h5ad"
GENE_ORDERED_H5AD = CNV_DIR / "GBM_core_4datasets_CNV_input_HVG8000_r0_6_allcells_gene_ordered_countsX.h5ad"
adata_cnv = adata.copy()
if "counts" in adata_cnv.layers:
    del adata_cnv.layers["counts"]
adata_cnv.uns["CNV_input_note"] = "Raw counts are stored in adata.X. Duplicate counts layer removed in this CNV export to reduce memory/disk usage. Original full-gene annotated object still preserves counts layer."
adata_cnv.uns["CNV_cluster_key"] = CLUSTER_KEY
adata_cnv.uns["CNV_main_query_clusters"] = ",".join(query_tumor_like)
adata_cnv.uns["CNV_core_reference_clusters"] = ",".join(core_reference)
write_h5ad_replace(adata_cnv, ALLCELLS_H5AD)
adata_go = adata_cnv[:, usable_genes].copy()
coord_map = infercnv_gene_order.set_index("gene")
adata_go.var["cnv_chr"] = coord_map.loc[adata_go.var_names, "chr"].astype(str).values
adata_go.var["cnv_start"] = coord_map.loc[adata_go.var_names, "start"].astype(int).values
adata_go.var["cnv_end"] = coord_map.loc[adata_go.var_names, "end"].astype(int).values
adata_go.uns["CNV_gene_order_file_header"] = str(GENE_MATCH_TSV)
adata_go.uns["CNV_gene_order_file_noheader"] = str(GENE_ORDER_NOHEADER_TSV)
write_h5ad_replace(adata_go, GENE_ORDERED_H5AD)
print("All-cell CNV exports complete")
print("allcells:", ALLCELLS_H5AD)
print("gene_ordered:", GENE_ORDERED_H5AD)


## 5. Export inferCNV-style cell annotations and sampled test subsets

For each strategy, the notebook exports:

- A cell annotation TSV with `cell_id` and `cnv_group`.
- A sampled h5ad with reference and query cells for a memory-safe test run.

The full all-cell object remains available for final larger runs.


In [ ]:
# =========================
# Export strategy-specific annotations and sampled h5ads
# =========================

strategy_manifest_rows = []
ref_group_rows = []
for strategy_name, info in strategies.items():
    ref_clusters = [str(x) for x in info["reference_clusters"]]
    query_clusters = [str(x) for x in info["query_clusters"]]
    obs = adata_go.obs.copy()
    cl = obs[CLUSTER_KEY].astype(str)
    ref_mask = cl.isin(ref_clusters)
    query_mask = cl.isin(query_clusters)
    keep_mask = ref_mask | query_mask
    if keep_mask.sum() == 0:
        print(f"[WARNING] No cells for strategy {strategy_name}; skipping")
        continue
    cnv_group = pd.Series(index=obs.index, dtype="object")
    cnv_group.loc[ref_mask] = "reference"
    cnv_group.loc[query_mask] = "query"
    ann = pd.DataFrame({
        "cell_id": obs.index.astype(str),
        "cnv_group": cnv_group.astype(str),
        "cluster": obs[CLUSTER_KEY].astype(str),
        "annotation": obs[ANNOTATION_COL].astype(str),
        "lineage": obs[LINEAGE_COL].astype(str),
        "condition": obs.get("condition", pd.Series("unknown", index=obs.index)).astype(str),
        "core_dataset": obs.get("core_dataset", pd.Series("unknown", index=obs.index)).astype(str),
        "patient_id": obs.get("patient_id", pd.Series("unknown", index=obs.index)).astype(str),
        "sample_id": obs.get("sample_id", pd.Series("unknown", index=obs.index)).astype(str),
    })
    ann = ann[keep_mask.values].copy()
    ann_path = METADATA_DIR / f"CNV_infercnv_cell_annotations_{strategy_name}_HVG8000_r0_6.tsv"
    write_tsv_replace(ann, ann_path)
    ref_names = ["reference"]
    ref_group_file = METADATA_DIR / f"CNV_reference_group_names_{strategy_name}_HVG8000_r0_6.tsv"
    write_text_replace(ref_names, ref_group_file)
    ref_group_rows.append({"strategy": strategy_name, "reference_group_name": "reference", "file": str(ref_group_file)})
    sampled_ref = sample_cells_by_cluster(adata_go.obs, ref_clusters, CLUSTER_KEY, MAX_REF_PER_STRATEGY, MAX_PER_CLUSTER_REF, RANDOM_SEED)
    sampled_query = sample_cells_by_cluster(adata_go.obs, query_clusters, CLUSTER_KEY, MAX_QUERY_PER_STRATEGY, MAX_PER_CLUSTER_QUERY, RANDOM_SEED + 1)
    sampled_cells = list(dict.fromkeys(sampled_ref + sampled_query))
    subset = adata_go[sampled_cells, :].copy()
    subset.obs["cnv_sampled_strategy"] = strategy_name
    subset.obs["cnv_group"] = np.where(subset.obs[CLUSTER_KEY].astype(str).isin(ref_clusters), "reference", "query")
    subset_path = CNV_DIR / f"GBM_core_4datasets_CNV_test_subset_{strategy_name}_HVG8000_r0_6_countsX.h5ad"
    write_h5ad_replace(subset, subset_path)
    strategy_manifest_rows.append({
        "strategy": strategy_name,
        "reference_clusters": ",".join(ref_clusters),
        "query_clusters": ",".join(query_clusters),
        "n_cells_annotation_file": ann.shape[0],
        "n_reference_cells_all": int(ref_mask.sum()),
        "n_query_cells_all": int(query_mask.sum()),
        "n_sampled_reference_cells": len(sampled_ref),
        "n_sampled_query_cells": len(sampled_query),
        "n_sampled_total_cells": len(sampled_cells),
        "annotation_tsv": str(ann_path),
        "reference_group_file": str(ref_group_file),
        "sampled_h5ad": str(subset_path),
    })
strategy_manifest = pd.DataFrame(strategy_manifest_rows)
ref_group_df = pd.DataFrame(ref_group_rows)
STRATEGY_MANIFEST_TSV = METADATA_DIR / "CNV_strategy_input_manifest_HVG8000_r0_6.tsv"
REF_GROUPS_TSV = METADATA_DIR / "CNV_reference_group_names_HVG8000_r0_6.tsv"
write_tsv_replace(strategy_manifest, STRATEGY_MANIFEST_TSV)
write_tsv_replace(ref_group_df, REF_GROUPS_TSV)
display(strategy_manifest)


## 6. Composition summaries for each CNV strategy

These summaries help evaluate whether each strategy is dominated by one dataset, condition, patient, or sample.


In [ ]:
# =========================
# Strategy composition summaries
# =========================

composition_rows = []
for strategy_name, info in strategies.items():
    ref_clusters = [str(x) for x in info["reference_clusters"]]
    query_clusters = [str(x) for x in info["query_clusters"]]
    cl = adata.obs[CLUSTER_KEY].astype(str)
    group = pd.Series("other", index=adata.obs.index)
    group.loc[cl.isin(ref_clusters)] = "reference"
    group.loc[cl.isin(query_clusters)] = "query"
    keep = group.isin(["reference", "query"])
    tmp = adata.obs.loc[keep, [CLUSTER_KEY, ANNOTATION_COL, LINEAGE_COL]].copy()
    tmp["strategy"] = strategy_name
    tmp["cnv_group"] = group.loc[keep].values
    for col in ["core_dataset", "condition", "patient_id", "sample_id"]:
        tmp[col] = adata.obs.loc[keep, col].astype(str).values if col in adata.obs.columns else "unknown"
    for group_name in ["reference", "query"]:
        sub = tmp[tmp["cnv_group"] == group_name]
        if sub.empty:
            continue
        row = {"strategy": strategy_name, "cnv_group": group_name, "n_cells": sub.shape[0], "n_clusters": sub[CLUSTER_KEY].nunique()}
        for col in ["core_dataset", "condition", "patient_id", "sample_id"]:
            vc = sub[col].value_counts(normalize=False)
            row[f"top_{col}"] = vc.index[0]
            row[f"top_{col}_n"] = int(vc.iloc[0])
            row[f"top_{col}_fraction"] = float(vc.iloc[0] / sub.shape[0])
            row[f"n_{col}"] = int(vc.shape[0])
        composition_rows.append(row)
composition_summary = pd.DataFrame(composition_rows)
COMPOSITION_TSV = METADATA_DIR / "CNV_strategy_composition_summary_HVG8000_r0_6.tsv"
write_tsv_replace(composition_summary, COMPOSITION_TSV)
display(composition_summary)


## 7. Final output manifest

In [ ]:
# =========================
# Output manifest
# =========================

manifest_items = [
    ("readiness_report", READINESS_TSV),
    ("cluster_category_counts", CLUSTER_CATEGORY_TSV),
    ("cell_category_counts", CELL_CATEGORY_TSV),
    ("strategy_definitions", STRATEGY_TSV),
    ("gene_position_report", GENE_POSITION_REPORT_TSV),
    ("gene_order_header", GENE_MATCH_TSV),
    ("gene_order_noheader", GENE_ORDER_NOHEADER_TSV),
    ("unmatched_genes", GENE_UNMATCHED_TSV),
    ("allcells_fullgene_countsX_h5ad", ALLCELLS_H5AD),
    ("allcells_gene_ordered_countsX_h5ad", GENE_ORDERED_H5AD),
    ("strategy_input_manifest", STRATEGY_MANIFEST_TSV),
    ("reference_group_names_summary", REF_GROUPS_TSV),
    ("strategy_composition_summary", COMPOSITION_TSV),
]
for _, row in strategy_manifest.iterrows():
    manifest_items.append((f"annotation_{row['strategy']}", Path(row["annotation_tsv"])))
    manifest_items.append((f"sampled_h5ad_{row['strategy']}", Path(row["sampled_h5ad"])))
manifest = pd.DataFrame([{
    "item": name,
    "path": str(path),
    "exists": Path(path).exists(),
    "size_mb": round(Path(path).stat().st_size / (1024 ** 2), 3) if Path(path).exists() else None,
} for name, path in manifest_items])
MANIFEST_TSV = METADATA_DIR / "CNV_input_output_manifest_HVG8000_r0_6.tsv"
write_tsv_replace(manifest, MANIFEST_TSV)
display(manifest)
print("CNV input preparation complete.")
print("Most important files to review:")
print(READINESS_TSV)
print(CELL_CATEGORY_TSV)
print(CLUSTER_CATEGORY_TSV)
print(GENE_POSITION_REPORT_TSV)
print(STRATEGY_MANIFEST_TSV)
print(COMPOSITION_TSV)
print(MANIFEST_TSV)
